### Connect to Database

In [1]:
import sqlite3

conn = sqlite3.connect("yt.db")
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = {row[0] for row in cur.fetchall()}
tables

{'comments', 'videos'}

In [2]:
has_videos = "videos" in tables
has_comments = "comments" in tables

print(f"videos table: {'✓' if has_videos else '✗'} found")
print(f"comments table: {'✓' if has_comments else '✗'} found")

videos table: ✓ found
comments table: ✓ found


In [3]:
if has_videos:
    cur.execute("PRAGMA table_info(videos)")
    print("videos columns:", [col[1] for col in cur.fetchall()])
if has_comments:
    cur.execute("PRAGMA table_info(comments)")
    print("comments columns:", [col[1] for col in cur.fetchall()])

videos columns: ['video_id', 'published_at', 'channel_id', 'title', 'description', 'category_id', 'view_count', 'like_count', 'dislike_count', 'comment_count', 'privacy_status', 'channel', 'fetched_comments_count']
comments columns: ['comment_id', 'video_id', 'thread_id', 'text_display', 'author_display_name', 'author_channel_id', 'like_count', 'published_at']


In [4]:
if has_videos:
    cur.execute("SELECT COUNT(*) FROM videos")
    print(f"videos row count: {cur.fetchone()[0]}")
if has_comments:
    cur.execute("SELECT COUNT(*) FROM comments")
    print(f"comments row count: {cur.fetchone()[0]}")

videos row count: 95
comments row count: 9984


# Analysis Questions 

## Q1: Channels with the Most Videos

In [5]:
if has_videos:
    cur.execute("""
        SELECT channel, channel_id, COUNT(*) AS video_count
        FROM videos
        GROUP BY channel
        ORDER BY video_count DESC
    """)
    rows = cur.fetchall()
    print(f"{'channel':<30} {'video_count':>12}")
    print("-" * 42)
    for row in rows:
        print(f"{row[0]:<30} {row[2]:>12}")

channel                         video_count
------------------------------------------
TechBar                                   3
Moksha_6_7                                3
TechRJ                                    2
Tech Master                               2
Phone Repair Guru                         2
Nick Ackerman                             2
MaybeHeilly                               2
JMTech                                    2
Isa Marcial                               2
Dev                                       2
CarterPCs                                 2
Beebom                                    2
AsmrLinastar                              2
Ahmed Sadoma | أحمد صَدُومة               2
ryan                                      1
pablo filters                             1
mobiscrub                                 1
iosAlex                                   1
iMAD Tech                                 1
cozytealeaf                               1
caitlin iola                     

## Q2: Videos with the Most Comments

In [6]:
if has_videos:
    cur.execute("""
        SELECT title, channel, comment_count
        FROM videos
        ORDER BY comment_count DESC
        LIMIT 20
    """)
    rows = cur.fetchall()
    print(f"{'title':<70} {'channel':<25} {'comments':>10}")
    print("-" * 107)
    for row in rows:
        print(f"{row[0][:68]:<70} {row[1]:<25} {row[2]:>10}")

title                                                                  channel                     comments
-----------------------------------------------------------------------------------------------------------
First IPhone 17 Pro Max unboxing ❤| JJ Communication                   JJ Communication               69128
iPhone 17 Pro Durability Test📱🪛                                        Money Saving Man               20373
iPhone 17 Pro Max Dropped In Public 🤯 #shorts                          Tech Master                     7196
جبت iphone 17 pro!!!                                                   Farah Roushdy                   6266
iPhone 17/Air/Pro Unboxing - What Apple didn't tell you!               Mrwhosetheboss                  6125
iPhone 17 Review: No Asterisks!                                        Marques Brownlee                5693
हमने किया Apple iPhone 17 Air का Bend Test📲 #shorts                    Tech Master                     5528
Iphone 17 Pro max Vs Samsung

## Q3: Commentors on the most videos

In [7]:
if has_comments:
    cur.execute("""
        SELECT author_display_name, author_channel_id, COUNT(DISTINCT video_id) AS video_count
        FROM comments
        GROUP BY author_display_name
        ORDER BY video_count DESC
        LIMIT 20
    """)
    rows = cur.fetchall()
    print(f"{'author':<35} {'video_count':>15}")
    print("-" * 52)
    for row in rows:
        print(f"{row[0][:33]:<35} {row[2]:>15}")

author                                  video_count
----------------------------------------------------
@kuwaitttissotuff                                 6
@queen-lolita                                     5
@Oppak95890                                       5
@nicoletiana                                      4
@fernando_1521                                    4
@brownandproud89                                  4
@bluesky5384                                      4
@TLG_NeXo                                         4
@SahilAzhar-b7e                                   4
@Jaquannsingletary0984                            4
@AlqaseemFamily                                   4
@wildflowercases                                  3
@strangevideos3048                                3
@rahulsindhwani4033                               3
@mehdiDn-f1s                                      3
@marinamekael1214                                 3
@jkrishnan30                                      3
@ellielovel

## Q4: Creators commenting on videos

In [8]:
if has_comments and has_videos:
    cur.execute("SELECT COUNT(DISTINCT author_channel_id) FROM comments WHERE author_channel_id IS NOT NULL AND author_channel_id != ''")
    print("Unique commenters:", cur.fetchone()[0])
    cur.execute("SELECT COUNT(DISTINCT channel_id) FROM videos")
    print("Unique video channels:", cur.fetchone()[0])
    cur.execute("""
        SELECT COUNT(DISTINCT c.author_channel_id)
        FROM comments c
        WHERE c.author_channel_id IS NOT NULL AND c.author_channel_id != ''
          AND c.author_channel_id IN (SELECT DISTINCT channel_id FROM videos)
    """)
    print("video channels that have commented on videos:", cur.fetchone()[0])

Unique commenters: 8758
Unique video channels: 79
video channels that have commented on videos: 39


## Q5: Channels Commenting on Other Channels' Videos

In [9]:
if has_comments and has_videos:
    cur.execute("""
        SELECT MAX(c.author_display_name), c.author_channel_id, COUNT(DISTINCT c.video_id) AS other_videos
        FROM comments c
        JOIN videos v ON c.video_id = v.video_id
        WHERE c.author_channel_id IS NOT NULL AND c.author_channel_id != ''
          AND c.author_channel_id != v.channel_id
          AND c.author_channel_id IN (SELECT DISTINCT channel_id FROM videos)
        GROUP BY c.author_channel_id
        ORDER BY other_videos DESC
        LIMIT 20
    """)
    rows = cur.fetchall()
    print(f"{'author':<35} {'other_videos':>15}")
    print("-" * 52)
    for row in rows:
        print(f"{row[0][:33]:<35} {row[2]:>15}")

author                                 other_videos
----------------------------------------------------


### Close Connection

In [10]:
conn.close()